# Giai đoạn 2: Tiền xử lý, chia dữ liệu và pipeline
Chỉ chuẩn bị dữ liệu và smoke test pipeline. Không tính metric và không huấn luyện mô hình thật.

In [2]:
# Nạp thư viện và module dự án
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

sys.path.insert(0, str(Path('..').resolve()))
from src.data_prep import load_data, clean_data, get_feature_sets
from src.pipeline import make_pipeline, choose_cv_and_k

RANDOM_STATE = 42
DATA_PATH = Path('../data/RT_IOT2022')
df_raw = load_data(DATA_PATH)
df_clean, clean_stats = clean_data(df_raw)
feature_sets = get_feature_sets(df_clean)
print(clean_stats)

{'rows_before': 123117, 'columns_before': 85, 'dropped_index_column': True, 'constant_columns': ['bwd_URG_flag_count'], 'duplicates_removed': 5195, 'rows_after': 117922, 'columns_after': 83}


## 0. Làm sạch dữ liệu
Đã bỏ cột chỉ số, cột hằng và các dòng trùng hoàn toàn. Đây là các bước đã chốt từ EDA để tránh dữ liệu lặp làm kết quả đánh giá quá lạc quan.

In [3]:
# So sánh phân bố lớp trước và sau loại trùng
before = df_raw['Attack_type'].value_counts().rename('truoc')
after = df_clean['Attack_type'].value_counts().rename('sau')
class_table = pd.concat([before, after], axis=1).fillna(0).astype(int)
class_table['giam_so_luong'] = class_table['truoc'] - class_table['sau']
class_table['giam_phan_tram'] = class_table['giam_so_luong'] / class_table['truoc'] * 100
display(class_table)
rare_after = class_table[class_table['sau'] < 20]
if not rare_after.empty:
    raise RuntimeError('DỪNG: có lớp còn dưới 20 mẫu sau khi loại trùng. Gửi bảng trên cho người hướng dẫn.')
print('Tình trạng lớp hiếm: không có lớp nào dưới 20 mẫu sau loại trùng.')

,truoc,sau,giam_so_luong,giam_phan_tram
Attack_type,,,,
DOS_SYN_Hping,94659,90089,4570,4.827856
Thing_Speak,8108,7654,454,5.599408
ARP_poisioning,7750,7625,125,1.612903
MQTT_Publish,4146,4142,4,0.096479
NMAP_UDP_SCAN,2590,2584,6,0.231660
NMAP_XMAS_TREE_SCAN,2010,2010,0,0.000000
NMAP_OS_DETECTION,2000,2000,0,0.000000
NMAP_TCP_scan,1002,1002,0,0.000000
DDOS_Slowloris,534,533,1,0.187266


Tình trạng lớp hiếm: không có lớp nào dưới 20 mẫu sau loại trùng.


Bảng trên cho biết mỗi lớp mất bao nhiêu dòng do trùng. Nếu có lớp dưới 20 mẫu, notebook chủ động dừng để không tiếp tục chia dữ liệu khi bằng chứng thống kê quá yếu.

In [4]:
# Đếm nhóm cùng đặc trưng nhưng khác nhãn
feature_only = df_clean.drop(columns=['Attack_type'])
group_sizes = feature_only.groupby(feature_only.columns.tolist(), dropna=False, sort=False).size()
conflicting = group_sizes[group_sizes > 1]
conflict_rows = df_clean.loc[feature_only.index.isin(conflicting.index if False else [])]
grouped = df_clean.groupby(feature_only.columns.tolist(), dropna=False, sort=False)['Attack_type'].agg(lambda s: tuple(sorted(s.unique())))
conflicting_labels = grouped[grouped.map(len) > 1]
print('Số nhóm cùng đặc trưng nhưng có nhiều nhãn:', len(conflicting_labels))
pairs = conflicting_labels.map(lambda x: tuple((x[i], x[j]) for i in range(len(x)) for j in range(i+1, len(x))))
pair_counts = pairs.explode().value_counts().rename('so_nhom')
display(pair_counts.to_frame())

Số nhóm cùng đặc trưng nhưng có nhiều nhãn: 6


,so_nhom
Attack_type,
"(ARP_poisioning, Thing_Speak)",4
"(ARP_poisioning, DDOS_Slowloris)",1
"(DDOS_Slowloris, Thing_Speak)",1
"(ARP_poisioning, NMAP_UDP_SCAN)",1
"(Metasploit_Brute_Force_SSH, NMAP_UDP_SCAN)",1


Các nhóm này chưa bị xóa. Chúng cho thấy cùng một vector đặc trưng có thể đi kèm nhiều nhãn, là tín hiệu cần phân tích khi đánh giá lỗi ở giai đoạn mô hình.

In [5]:
# Chia stratified 80/20 và lưu chỉ số
indices = np.arange(len(df_clean))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=RANDOM_STATE, stratify=df_clean['Attack_type'])
train_idx = np.sort(train_idx); test_idx = np.sort(test_idx)
Path('../experiments').mkdir(exist_ok=True)
np.savez('../experiments/split_indices.npz', train_idx=train_idx, test_idx=test_idx)
assert set(train_idx).isdisjoint(set(test_idx))
assert len(train_idx) + len(test_idx) == len(df_clean)
split_table = pd.DataFrame({'train': df_clean.iloc[train_idx]['Attack_type'].value_counts(), 'test': df_clean.iloc[test_idx]['Attack_type'].value_counts()}).fillna(0).astype(int)
display(split_table)
print('Train/test không giao nhau: True')
min_train = int(split_table['train'].min())
cv_folds, k_neighbors = choose_cv_and_k(min_train)
print(f'Lớp nhỏ nhất train: {min_train}; số fold CV: {cv_folds}; k_neighbors SMOTE: {k_neighbors}')

,train,test
Attack_type,,
DOS_SYN_Hping,72071,18018
Thing_Speak,6123,1531
ARP_poisioning,6100,1525
MQTT_Publish,3314,828
NMAP_UDP_SCAN,2067,517
NMAP_XMAS_TREE_SCAN,1608,402
NMAP_OS_DETECTION,1600,400
NMAP_TCP_scan,802,200
DDOS_Slowloris,426,107


Train/test không giao nhau: True
Lớp nhỏ nhất train: 22; số fold CV: 5; k_neighbors SMOTE: 3


Dữ liệu được chia theo nhãn để tỷ lệ lớp được giữ ở train và test. Chỉ số được lưu riêng, không lưu bản sao dữ liệu. Số fold CV và `k_neighbors` được tính từ lớp nhỏ nhất của train, không đoán trước.

In [6]:
# Tạo và smoke test pipeline trên train
X = df_clean.drop(columns=['Attack_type'])
y = df_clean['Attack_type']
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
pipe = make_pipeline(DummyClassifier(strategy='most_frequent'), scale=False, sampler=None, feature_columns=X_train.columns.tolist())
print(pipe)
pipe.fit(X_train, y_train)
_ = pipe.predict(X_test)
print('Smoke test thành công: fit chỉ trên X_train/y_train; test chỉ được transform/predict.')
print('Không tính metric và không ghi kết quả smoke test vào experiment log.')

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  <function make_pipeline_from_columns.<locals>.categorical_cols at 0x00000273AB6F09A0>),
                                                 ('numeric', 'passthrough',
                                                  <function make_pipeline_from_columns.<locals>.numeric_cols at 0x000002741A59A980>)])),
                ('model', DummyClassifier(strategy='most_frequent'))])
Smoke test thành công: fit chỉ trên X_train/y_train; test chỉ được transform/predict.
Không tính metric và không ghi kết quả smoke test vào experiment log.


Pipeline chứa encoder trong `ColumnTransformer`, bước lấy mẫu tùy chọn và model cuối. Smoke test chỉ kiểm tra pipeline chạy được; lệnh predict trên test không được dùng để chọn mô hình hay tính metric.

## Bộ đặc trưng đã chốt
`FULL` giữ tất cả đặc trưng sau làm sạch; `NO_SRC_PORT` bỏ `id.orig_p` vì cổng nguồn có rất nhiều giá trị và dễ làm mô hình học thuộc phiên/kịch bản; `NO_PORTS` bỏ thêm `id.resp_p` để làm đối chứng rò rỉ. Bộ chính là `NO_SRC_PORT`, còn hai bộ kia dùng so sánh ở giai đoạn mô hình.

In [7]:
# In các bộ đặc trưng và tổng kết có số liệu thật
for name, cols in feature_sets.items(): print(name, len(cols), 'đặc trưng')
print('=== TỔNG KẾT GIAI ĐOẠN 2 ===')
print(f'Số dòng trước làm sạch: {clean_stats["rows_before"]}; sau làm sạch: {clean_stats["rows_after"]}.')
print(f'Đã bỏ cột hằng: {clean_stats["constant_columns"]}; số dòng trùng đã bỏ: {clean_stats["duplicates_removed"]}.')
print('Số mẫu từng lớp train/test:')
display(split_table)
print(f'Lớp nhỏ nhất train: {min_train}; CV: {cv_folds} fold; SMOTE k_neighbors: {k_neighbors}.')
print('Tình trạng lớp hiếm: không có lớp nào dưới 20 mẫu sau loại trùng; lớp nhỏ nhất train vẫn được dùng để giới hạn CV và SMOTE.')
print('Quyết định: dùng NO_SRC_PORT làm bộ chính; giữ FULL và NO_PORTS làm đối chứng; chưa xóa nhóm cùng đặc trưng khác nhãn.')

FULL 82 đặc trưng
NO_SRC_PORT 81 đặc trưng
NO_PORTS 80 đặc trưng
=== TỔNG KẾT GIAI ĐOẠN 2 ===
Số dòng trước làm sạch: 123117; sau làm sạch: 117922.
Đã bỏ cột hằng: ['bwd_URG_flag_count']; số dòng trùng đã bỏ: 5195.
Số mẫu từng lớp train/test:


,train,test
Attack_type,,
DOS_SYN_Hping,72071,18018
Thing_Speak,6123,1531
ARP_poisioning,6100,1525
MQTT_Publish,3314,828
NMAP_UDP_SCAN,2067,517
NMAP_XMAS_TREE_SCAN,1608,402
NMAP_OS_DETECTION,1600,400
NMAP_TCP_scan,802,200
DDOS_Slowloris,426,107


Lớp nhỏ nhất train: 22; CV: 5 fold; SMOTE k_neighbors: 3.
Tình trạng lớp hiếm: không có lớp nào dưới 20 mẫu sau loại trùng; lớp nhỏ nhất train vẫn được dùng để giới hạn CV và SMOTE.
Quyết định: dùng NO_SRC_PORT làm bộ chính; giữ FULL và NO_PORTS làm đối chứng; chưa xóa nhóm cùng đặc trưng khác nhãn.


## Ô tổng kết cuối Giai đoạn 2
Ô code kế tiếp in số liệu thật và quyết định đã chốt.

In [9]:
# In tổng kết cuối có số liệu thật
print('=== TỔNG KẾT GIAI ĐOẠN 2 ===')
print(f'Số dòng trước làm sạch: {clean_stats["rows_before"]}; sau làm sạch: {clean_stats["rows_after"]}.')
print(f'Đã bỏ cột hằng: {clean_stats["constant_columns"]}; số dòng trùng đã bỏ: {clean_stats["duplicates_removed"]}.')
print('Số mẫu từng lớp train/test:')
display(split_table)
print(f'Lớp nhỏ nhất train: {min_train}; CV: {cv_folds} fold; SMOTE k_neighbors: {k_neighbors}.')
print('Tình trạng lớp hiếm: không có lớp nào dưới 20 mẫu sau loại trùng; lớp nhỏ nhất train được dùng để giới hạn CV và SMOTE.')
print('Quyết định: dùng NO_SRC_PORT làm bộ chính; giữ FULL và NO_PORTS làm đối chứng; chưa xóa nhóm cùng đặc trưng khác nhãn.')

=== TỔNG KẾT GIAI ĐOẠN 2 ===
Số dòng trước làm sạch: 123117; sau làm sạch: 117922.
Đã bỏ cột hằng: ['bwd_URG_flag_count']; số dòng trùng đã bỏ: 5195.
Số mẫu từng lớp train/test:


,train,test
Attack_type,,
DOS_SYN_Hping,72071,18018
Thing_Speak,6123,1531
ARP_poisioning,6100,1525
MQTT_Publish,3314,828
NMAP_UDP_SCAN,2067,517
NMAP_XMAS_TREE_SCAN,1608,402
NMAP_OS_DETECTION,1600,400
NMAP_TCP_scan,802,200
DDOS_Slowloris,426,107


Lớp nhỏ nhất train: 22; CV: 5 fold; SMOTE k_neighbors: 3.
Tình trạng lớp hiếm: không có lớp nào dưới 20 mẫu sau loại trùng; lớp nhỏ nhất train được dùng để giới hạn CV và SMOTE.
Quyết định: dùng NO_SRC_PORT làm bộ chính; giữ FULL và NO_PORTS làm đối chứng; chưa xóa nhóm cùng đặc trưng khác nhãn.
